# 01 — Hospital Data Exploration

**Purpose:** Explore the Australian Institute of Health and Welfare (AIHW) MyHospitals API before building the hospital performance pipeline.

This notebook documents the source-data investigation: available measure metadata, the structure of observations and datasets, reporting-period and reporting-unit coverage, and how missing values and suppression details are represented.

**Scope:** Uses `MYH0005` (four-hour emergency department departure rate) as a sample measure. This is an exploratory notebook; the production ingestion and transformation steps are handled by the Bronze, Silver, and Gold notebooks.

In [ ]:
import json
import requests
from collections import Counter

from pyspark.sql.types import (
    StructType, StructField, StringType
)

base_url = "https://myhospitalsapi.aihw.gov.au/api/v1"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

In [ ]:
# Check the available measure categories and measures

categories_response = requests.get(
    f"{base_url}/measure-categories",
    headers=headers,
    timeout=30
)
categories_response.raise_for_status()
categories = categories_response.json().get("result", [])

measures_response = requests.get(
    f"{base_url}/measures",
    headers=headers,
    timeout=30
)
measures_response.raise_for_status()
measures = measures_response.json().get("result", [])

print("Measure categories:", len(categories))
print("Measures:", len(measures))

display(spark.createDataFrame(categories))
display(spark.createDataFrame(measures))

In [ ]:
# Inspect the measure metadata fields and their data types

if measures:
    for field, value in measures[0].items():
        print(f"{field}: {type(value).__name__} — {repr(value)[:160]}")

measure_schema = StructType([
    StructField("measure_code", StringType(), True),
    StructField("measure_name", StringType(), True),
    StructField("measure_categories_json", StringType(), True),
    StructField("meta_tags_json", StringType(), True),
    StructField("units_json", StringType(), True)
])

measure_rows = [
    (
        str(measure.get("measure_code") or ""),
        str(measure.get("measure_name") or ""),
        json.dumps(measure.get("measure_categories") or []),
        json.dumps(measure.get("meta_tags") or []),
        json.dumps(measure.get("units") or {})
    )
    for measure in measures
]

measures_df = spark.createDataFrame(measure_rows, schema=measure_schema)
print("Measures loaded:", measures_df.count())
measures_df.select("measure_code", "measure_name").show(40, truncate=False)

In [ ]:
# Retrieve one measure's observations to understand their structure

measure_code = "MYH0005"
data_items_response = requests.get(
    f"{base_url}/measures/{measure_code}/data-items",
    headers=headers,
    timeout=30
)
data_items_response.raise_for_status()
data_items = data_items_response.json().get("result", [])

print(f"{measure_code} data items:", len(data_items))

if data_items:
    print("Fields in an observation:")
    print(list(data_items[0].keys()))
    print("\nSample observation:")
    print(json.dumps(data_items[0], indent=2)[:3000])

In [ ]:
# Retrieve dataset metadata and check how observations link to datasets

datasets_response = requests.get(
    f"{base_url}/datasets",
    headers=headers,
    timeout=30
)
datasets_response.raise_for_status()
datasets = datasets_response.json().get("result", [])

dataset_ids = [dataset.get("data_set_id") for dataset in datasets]
measure_dataset_ids = {
    item.get("data_set_id") for item in data_items
}

matching_measure_datasets = [
    dataset for dataset in datasets
    if dataset.get("data_set_id") in measure_dataset_ids
]
matched_dataset_ids = {
    dataset.get("data_set_id") for dataset in matching_measure_datasets
}

print("Dataset records:", len(datasets))
print("Unique dataset IDs:", len(set(dataset_ids)))
print(f"Distinct dataset IDs in {measure_code}:", len(measure_dataset_ids))
print("Matched dataset records:", len(matching_measure_datasets))
print("Observation dataset IDs without metadata:", len(
    measure_dataset_ids - matched_dataset_ids
))

for dataset in matching_measure_datasets[:10]:
    summary = dataset.get("reported_measure_summary") or {}
    print({
        "data_set_id": dataset.get("data_set_id"),
        "data_set_name": dataset.get("data_set_name"),
        "reported_measure_code": summary.get("reported_measure_code"),
        "reported_measure_name": summary.get("reported_measure_name"),
        "reporting_start_date": dataset.get("reporting_start_date"),
        "reporting_end_date": dataset.get("reporting_end_date")
    })

In [ ]:
# Review reporting periods, reported-measure variants, and reporting units

period_counts = Counter()
variant_counts = Counter()
unit_type_counts = Counter()
value_counts = Counter()
reported_measure_counts = Counter()

for dataset in matching_measure_datasets:
    period = (
        dataset.get("reporting_start_date"),
        dataset.get("reporting_end_date")
    )
    period_counts[period] += 1

    summary = dataset.get("reported_measure_summary") or {}
    variant_counts[summary.get("reported_measure_name", "Unknown")] += 1

for item in data_items:
    unit = item.get("reporting_unit_summary") or {}
    unit_type = unit.get("reporting_unit_type") or {}

    unit_type_counts[
        unit_type.get("reporting_unit_type_name", "Unknown")
    ] += 1
    value_counts[
        "Null value" if item.get("value") is None else "Non-null value"
    ] += 1
    reported_measure_counts[
        item.get("reported_measure_code", "Unknown")
    ] += 1

print("DATASETS BY REPORTING PERIOD")
for period, count in sorted(period_counts.items()):
    print(f"{period[0]} to {period[1]}: {count}")

print("\nDATASETS BY REPORTED-MEASURE VARIANT")
for name, count in variant_counts.most_common():
    print(f"{name}: {count}")

print("\nOBSERVATIONS BY REPORTING-UNIT TYPE")
for name, count in unit_type_counts.most_common():
    print(f"{name}: {count}")

print("\nVALUE AVAILABILITY")
for name, count in value_counts.items():
    print(f"{name}: {count}")

print("\nOBSERVATIONS BY REPORTED-MEASURE CODE")
for code, count in reported_measure_counts.most_common():
    print(f"{code}: {count}")

In [ ]:
# Check null observations and the suppression details attached to them

null_items = [
    item for item in data_items
    if item.get("value") is None
]

suppression_counts = Counter()
unspecified_suppression_count = 0

for item in null_items:
    suppressions = item.get("suppressions") or []

    if not suppressions:
        unspecified_suppression_count += 1
        continue

    for suppression in suppressions:
        key = (
            suppression.get("suppression_code", "Unknown"),
            suppression.get("suppression_name", "Unknown")
        )
        suppression_counts[key] += 1

print("Null-value observations:", len(null_items))
print("Null observations without suppression details:",
      unspecified_suppression_count)

print("\nSUPPRESSION REASONS")
for (code, name), count in suppression_counts.most_common():
    print(f"{code} - {name}: {count}")

print("\nSAMPLE NULL OBSERVATIONS")
for item in null_items[:5]:
    unit = item.get("reporting_unit_summary") or {}
    print({
        "data_set_id": item.get("data_set_id"),
        "reported_measure_code": item.get("reported_measure_code"),
        "reporting_unit": unit.get("reporting_unit_name"),
        "suppressions": item.get("suppressions"),
        "caveats": item.get("caveats")
    })